## Limpieza de 'DF_SPORTMANIAC_SUCIO.csv'

Cargamos la tabla de carreras/modalidades (una fila por cada modalidad de cada evento) y le hacemos una primera inspección antes de limpiarla, siguiendo el mismo proceso que en buscametas/xipgroc/carreirasgalegas/ccnorte/cronofinisher/mychip.

A diferencia de las demás fuentes, esta tabla incluye eventos de todo el mundo (SportManiacs no es una plataforma solo española), así que el primer filtro es quedarnos solo con `pais == "España"`.

In [1]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("../../data/raw/sportmaniacs/DF_SPORTMANIACS_SUCIO.csv")
curses = pd.read_csv(CSV_PATH, encoding="utf-8-sig", low_memory=False)

print("Filas x columnas:", curses.shape)
print()
print(curses.dtypes)
curses.head()

Filas x columnas: (44114, 25)

id                                object
idRace                             int64
nom                               object
slug                              object
data                              object
ciutat                            object
provincia                         object
pais                              object
email                             object
telefon                           object
descripcio                        object
data_final                        object
latitude                         float64
longitude                        float64
reglament                         object
num_fitxers                      float64
showRankings                        bool
active_photos                       bool
externalInscriptions              object
event_id                          object
distancia                         object
participantes_totales            float64
participantes_hombres            float64
participantes_mujeres     

,id,idRace,nom,slug,data,ciutat,provincia,pais,email,telefon,...,num_fitxers,showRankings,active_photos,externalInscriptions,event_id,distancia,participantes_totales,participantes_hombres,participantes_mujeres,participantes_sin_especificar
0,5d3acf92-6f08-4a5a-8593-016fac1f0671,860000200008,III PODOACTIVA MEDIEVAL TRAIL MONTEARAGON,iii-podoactiva-medieval-trail-montearagn,2019-09-14,Quicena,Huesca,España,contacto@sportevento.com,657555740,...,3.0,True,False,NaN,5d3acf93-0754-4f56-9579-016fac1f0671,TRAIL 21K,54.0,49.0,5.0,0.0
1,5d3acf92-6f08-4a5a-8593-016fac1f0671,860000200008,III PODOACTIVA MEDIEVAL TRAIL MONTEARAGON,iii-podoactiva-medieval-trail-montearagn,2019-09-14,Quicena,Huesca,España,contacto@sportevento.com,657555740,...,3.0,True,False,NaN,5d3acfe5-c1c4-4ffb-802c-0420ac1f0671,TRAIL 9K,80.0,65.0,15.0,0.0
2,5d3add4a-b684-4773-b300-17b9ac1f0671,840000300003,DH LA MONTAÑA R. FINAL,dh-la-montatildea-r-final,2019-07-28,Realejos (Los),Santa Cruz de Tenerife,España,mdsports.eventos@gmail.com,685131284,...,0.0,False,False,NaN,5d3addd4-92a0-4479-85af-0170ac1f0671,FINAL,81.0,81.0,0.0,0.0
3,5d3b120f-7110-4520-8714-2f0dac1f0671,10000100754,Marxa Bonesvalls 2019,marxa-bonesvalls-2019,2019-10-20,Olesa de Bonesvalls,Barcelona,España,marxabonesvalls@gmail.com,NaN,...,1.0,False,False,NaN,5dac44b1-dc54-4946-8f70-1edeac1f00b3,MB30,65.0,50.0,15.0,0.0
4,5d3b120f-7110-4520-8714-2f0dac1f0671,10000100754,Marxa Bonesvalls 2019,marxa-bonesvalls-2019,2019-10-20,Olesa de Bonesvalls,Barcelona,España,marxabonesvalls@gmail.com,NaN,...,1.0,False,False,NaN,5dac44b8-e6d0-4ee9-83cb-4f22ac1f1f76,MB14,250.0,153.0,97.0,0.0


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados, países y estado de los
# datos. "event_id"/"distancia"/los "participantes_*" son NaN cuando el
# scraper no llegó a sacar ningún dato de resultados para ese evento
# (igual que "sense_resultats" en cronofinisher) — se comprueba más abajo.
print("Valores nulos por columna:")
print(curses.isna().sum())
print()

print("Filas completamente duplicadas:", curses.duplicated().sum())
print()

print("Rango de fechas (texto):", curses["data"].min(), "->", curses["data"].max())
print()

print("Países (top 15):")
print(curses["pais"].value_counts().head(15))
print()

print("Filas sin event_id (sin datos de resultados):", curses["event_id"].isna().sum())
print("De esas, con participantes_totales > 0:",
      (curses.loc[curses["event_id"].isna(), "participantes_totales"].fillna(0) > 0).sum())

Valores nulos por columna:
id                                   0
idRace                               0
nom                                  0
slug                                 0
data                                 0
ciutat                              21
provincia                            0
pais                                 0
email                              938
telefon                          21595
descripcio                       43015
data_final                           0
latitude                         15569
longitude                        15569
reglament                        43190
num_fitxers                          0
showRankings                         0
active_photos                        0
externalInscriptions             43692
event_id                          9778
distancia                         9778
participantes_totales               66
participantes_hombres               66
participantes_mujeres               66
participantes_sin_especificar       6

In [3]:
# Filtramos a solo España (SportManiacs incluye eventos de medio mundo:
# México, Francia, Polonia...) y descartamos las filas sin "event_id" —
# el diagnóstico de arriba confirma que esas filas nunca tienen
# participantes (0 o NaN), así que no aportan ningún dato de resultados.
antes = len(curses)
curses = curses[curses["pais"] == "España"].reset_index(drop=True)
print(f"Filas fuera de España descartadas: {antes - len(curses)} ({antes} -> {len(curses)})")

antes = len(curses)
curses = curses[curses["event_id"].notna()].reset_index(drop=True)
print(f"Filas sin datos de resultados descartadas: {antes - len(curses)} ({antes} -> {len(curses)})")

# Quitamos duplicados exactos ANTES de seleccionar columnas — con "event_id"
# todavía presente (el identificador único real de cada modalidad). Es el
# mismo criterio que en xipgroc/carreirasgalegas/cronofinisher/mychip: hacerlo
# después de descartar columnas identificadoras podría confundir filas
# legítimas y distintas con duplicados.
antes = len(curses)
curses = curses.drop_duplicates().reset_index(drop=True)
print(f"{antes - len(curses)} filas duplicadas eliminadas ({antes} -> {len(curses)})")

Filas fuera de España descartadas: 15979 (44114 -> 28135)
Filas sin datos de resultados descartadas: 7291 (28135 -> 20844)


0 filas duplicadas eliminadas (20844 -> 20844)


In [4]:
# Limpieza: nos quedamos solo con las columnas que interesan, renombradas.
# "ciutat"/"provincia" ya vienen directos de la fuente (municipio limpio,
# sin ruido). "distancia" en origen es en realidad texto de modalidad
# (mezcla distancia/edad/género, p.ej. "10K", "SUB 14", "TRAIL"), así que
# la guardamos como "modalidad" y calculamos la "distancia" real (km) más
# abajo, igual que hicimos con "modalitat_nom" en cronofinisher.
curses_limpio = curses[
    ["nom", "data", "ciutat", "provincia", "distancia",
     "participantes_mujeres", "participantes_hombres",
     "participantes_sin_especificar", "event_id"]
].rename(columns={
    "nom": "nombre_carrera",
    "data": "fecha",
    "ciutat": "municipio",
    "distancia": "modalidad",
    "participantes_mujeres": "finisher_d",
    "participantes_hombres": "finisher_h",
    "participantes_sin_especificar": "finisher_desconocido",
    "event_id": "id",
})

curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha"])
curses_limpio[["finisher_d", "finisher_h", "finisher_desconocido"]] = (
    curses_limpio[["finisher_d", "finisher_h", "finisher_desconocido"]].fillna(0).astype(int)
)

print(curses_limpio.shape)
curses_limpio.head()

(20844, 9)


,nombre_carrera,fecha,municipio,provincia,modalidad,finisher_d,finisher_h,finisher_desconocido,id
0,III PODOACTIVA MEDIEVAL TRAIL MONTEARAGON,2019-09-14,Quicena,Huesca,TRAIL 21K,5,49,0,5d3acf93-0754-4f56-9579-016fac1f0671
1,III PODOACTIVA MEDIEVAL TRAIL MONTEARAGON,2019-09-14,Quicena,Huesca,TRAIL 9K,15,65,0,5d3acfe5-c1c4-4ffb-802c-0420ac1f0671
2,DH LA MONTAÑA R. FINAL,2019-07-28,Realejos (Los),Santa Cruz de Tenerife,FINAL,0,81,0,5d3addd4-92a0-4479-85af-0170ac1f0671
3,Marxa Bonesvalls 2019,2019-10-20,Olesa de Bonesvalls,Barcelona,MB30,15,50,0,5dac44b1-dc54-4946-8f70-1edeac1f00b3
4,Marxa Bonesvalls 2019,2019-10-20,Olesa de Bonesvalls,Barcelona,MB14,97,153,0,5dac44b8-e6d0-4ee9-83cb-4f22ac1f1f76


In [5]:
# Clasificamos la disciplina por palabras clave en "modalidad": si no da
# ninguna pista, probamos con el nombre de la carrera antes de rendirnos.
# Deportes de una sola modalidad ajena a nuestras categorías (natación/
# travesías a nado, canicross, obstáculos, adaptado) van a "Otros", igual
# que "swimming"/"climbing" en mychip. "road running" ahora tiene su propio
# patrón (antes no lo tenía y era el valor por defecto para cualquier cosa
# sin reconocer, lo que metía ahí etiquetas de solo edad/formato como
# "INFANTIL" o "Sprint" sin ninguna prueba real de que fueran running de
# asfalto) — "Sprint"/"Supersprint" es el nombre habitual del formato corto
# de triatlón/duatlón, así que se van a Multidisciplina; lo que queda sin
# ninguna pista (edad sola, "CORTA"/"LARGA"...) cae en "Otros", igual que
# en buscametas/ccnorte, y solo se asume "road running" por defecto cuando
# no hay ningún texto en absoluto.
#
# Excepción por nombre de carrera: "Mallorca 312" y "SB Hotels La Garba"
# son marchas cicloturistas/gran fondo reales (167-343 km, con variante
# "Ebike" en La Garba) pero ni la modalidad ("167KM", "LA GARBA 123K"...)
# ni el nombre del evento contienen ninguna palabra clave de ciclismo
# ("btt"/"mtb"/"bike"/"ciclis"/"bici"/"gran fondo"/"gravel"/"ciclotur") —
# solo el kilometraje, que además dispara el patrón "\d+\s?km?\b" de
# "road running". No hay ninguna palabra clave genérica que capture esto
# sin arriesgar falsos positivos en otras carreras, así que se listan las
# dos por nombre en vez de forzar un patrón nuevo.
import re

_EXCEPCIONES_NOMBRE_CICLISMO = r"mallorca 312|sb hotels la garba"

_CATEGORIAS = {
    "Multidisciplina": r"triatl|duatl|swimrun|ol[ií]mpic|aquatl|acuatl|\betapa\b|\bstage\b|multidep|multidisci|combinada|\btc\d\b|sprint",
    "trail running": r"trail|\bcross\b|kv\b|vertical|trekking",
    "Ciclismo y btt": r"\bbtt\b|\bmtb\b|\bbike\b|ciclis|\bbici\b|gran ?fondo|gravel|ciclotur|e-?bikes?",
    "marcha": r"marcha|marxa|caminad|andarin|walking|nordic|\brelleus\b",
    "Otros": r"\bnado\b|natac|\bswim\b|travesi|travess|canicross|obst[aá]cul|\bocr\b|discapac|handbike|silla de ruedas",
    "road running": r"carrera|corre|running|popular|asfalto|ruta|marat|absoluta|corredor|legua|general|\bopen\b|\d+\s?km?\b",
}

def _clasificar_texto(texto):
    for categoria, patron in _CATEGORIAS.items():
        if re.search(patron, texto, flags=re.IGNORECASE):
            return categoria
    return None

def _clasificar(row):
    texto_nombre = row["nombre_carrera"] if isinstance(row["nombre_carrera"], str) else ""
    if re.search(_EXCEPCIONES_NOMBRE_CICLISMO, texto_nombre, flags=re.IGNORECASE):
        return "Ciclismo y btt"

    texto_modalidad = row["modalidad"] if isinstance(row["modalidad"], str) else ""
    categoria = _clasificar_texto(texto_modalidad) if texto_modalidad else None
    if categoria:
        return categoria

    categoria = _clasificar_texto(texto_nombre)
    if categoria:
        return categoria

    if texto_modalidad == "":
        return "road running"
    return "Otros"

curses_limpio["tipo_modalidad"] = curses_limpio.apply(_clasificar, axis=1)

print(curses_limpio["tipo_modalidad"].value_counts())

tipo_modalidad
road running       9801
Otros              4618
trail running      2466
Multidisciplina    2085
Ciclismo y btt     1287
marcha              587
Name: count, dtype: int64


In [6]:
# Clasificamos el público (edad) por palabras clave, con SubXX llevando la
# edad directamente en el número (≤12 Infantil, 13-23 Cadete/Juvenil,
# igual que en xipgroc/cronofinisher/mychip). Discapacidad/handbike/silla
# de ruedas/adaptado son categorías especiales, no de edad, así que van a
# "Otros" (comprobado antes que nada, para no confundir p.ej. "SUB18 A
# VETERANOS Y DISCAPACITADOS" con una categoría de edad normal). Si no hay
# ninguna marca de edad ni es especial, asumimos Absoluta/General por
# defecto (incluye las etiquetas de solo distancia/género, como "10K" o
# "MASCULINOS", que no dan ninguna pista de edad).
_OTROS_PATRON = r"discapac|invident|handbike|hand bike|silla de ruedas|adaptad|paral[ií]mpic"
_EQUIPOS_PATRON = r"equipos?\b|equips?\b"

_PUBLICOS_TEXTO = {
    "Elite": r"\belite\b|[ée]lite|profesional",
    "Mayores/Veteranos": r"veteran|master|m[aá]ster",
}
_INFANTIL_PATRON = (
    r"infantil|alev[ií]n|benjam[ií]n|prebenjam[ií]n|baby|escolar|menores|ni[ñn]os|"
    r"chupet|pitufo|querub[ií]n|minibenjam"
)
_CADETE_PATRON = r"cadete|juvenil|junior|j[uú]nior|promesa"

def _clasificar_publico_texto(texto):
    if re.search(_EQUIPOS_PATRON, texto, flags=re.IGNORECASE):
        return "Equipos"

    if re.search(_OTROS_PATRON, texto, flags=re.IGNORECASE):
        return "Otros"

    for publico, patron in _PUBLICOS_TEXTO.items():
        if re.search(patron, texto, flags=re.IGNORECASE):
            return publico

    _sb = re.search(r"\bsub[\s-]?(\d{1,2})\b", texto, flags=re.IGNORECASE)
    if _sb:
        edad = int(_sb.group(1))
        if edad <= 12:
            return "Infantil"
        if edad <= 23:
            return "Cadete/Juvenil"

    if re.search(_INFANTIL_PATRON, texto, flags=re.IGNORECASE):
        return "Infantil"

    if re.search(_CADETE_PATRON, texto, flags=re.IGNORECASE):
        return "Cadete/Juvenil"

    return None

def _clasificar_publico(row):
    texto_modalidad = row["modalidad"] if isinstance(row["modalidad"], str) else ""
    publico = _clasificar_publico_texto(texto_modalidad) if texto_modalidad else None
    if publico:
        return publico

    texto_nombre = row["nombre_carrera"] if isinstance(row["nombre_carrera"], str) else ""
    publico = _clasificar_publico_texto(texto_nombre)
    if publico:
        return publico

    return "Absoluta/General"

curses_limpio["publico"] = curses_limpio.apply(_clasificar_publico, axis=1)

print(curses_limpio["publico"].value_counts())
print()
print("Texto de modalidad clasificado como Otros (categorías especiales):")
print(curses_limpio.loc[curses_limpio["publico"] == "Otros", "modalidad"].value_counts().head(20))

publico
Absoluta/General     16097
Infantil              2540
Cadete/Juvenil        1199
Mayores/Veteranos      297
Otros                  286
Equipos                237
Elite                  188
Name: count, dtype: int64

Texto de modalidad clasificado como Otros (categorías especiales):
modalidad
HANDBIKE                                                           20
HandBike                                                           14
Maratón Silla de Ruedas                                             8
Silla de Ruedas                                                     6
HandBikes                                                           6
Hand Bike                                                           6
HANDBIKE provisional                                                6
CARRERA 3 - SUB18 A VETERANOS F Y DISCAPACITADOS                    5
SUB16 MASC-SUB20 FEM-VET FEM Y DISCAPACITADA FEM                    5
CARRERA VETERANOS Y DISCAPACITADOS                                  5

In [7]:
# Extraemos la distancia (km) donde se pueda del texto de "modalidad":
# K/KM (con o sin espacio/punto/mayúsculas), maratón/media maratón/milla
# con su distancia oficial. La mayoría de valores son solo categorías de
# edad/género sin ninguna distancia (p.ej. "INFANTIL", "ABSOLUTA"), así
# que se quedan en 0 — no hay forma fiable de inferirla.
_km = curses_limpio["modalidad"].str.extract(r"(\d+(?:[.,]\d+)?)\s*k(?:m)?\.?\b", flags=re.IGNORECASE)[0]
distancia_km = _km.str.replace(",", ".", regex=False).astype(float)

_es_media = curses_limpio["modalidad"].str.contains(
    r"media\s*marat|medio\s*marat|mitja\s*marat", case=False, regex=True, na=False
)
_es_marat = curses_limpio["modalidad"].str.contains(r"marat", case=False, regex=True, na=False)
_es_milla = curses_limpio["modalidad"].str.contains(r"\bmilla\b", case=False, regex=True, na=False)

curses_limpio["distancia"] = distancia_km
_falta = curses_limpio["distancia"].isna()
curses_limpio.loc[_falta & _es_media, "distancia"] = 21.097
_falta = curses_limpio["distancia"].isna()
curses_limpio.loc[_falta & _es_marat & ~_es_media, "distancia"] = 42.195
_falta = curses_limpio["distancia"].isna()
curses_limpio.loc[_falta & _es_milla, "distancia"] = 1.609

curses_limpio["distancia"] = curses_limpio["distancia"].fillna(0)

print("Filas con distancia detectada:", (curses_limpio["distancia"] != 0).sum(),
      "de", len(curses_limpio))
print()
print("Ejemplos de modalidad SIN distancia detectada (categorías de edad/género/disciplina sin número):")
print(curses_limpio.loc[curses_limpio["distancia"] == 0, "modalidad"].value_counts().head(20))

Filas con distancia detectada: 7106 de 20844

Ejemplos de modalidad SIN distancia detectada (categorías de edad/género/disciplina sin número):
modalidad
ABSOLUTA      210
GENERAL       134
Absoluta      122
TRAIL         122
INFANTIL       85
RUTA CORTA     65
FEMENINAS      64
Sprint         64
MASCULINOS     64
CARRERA        60
RUTA LARGA     57
INDIVIDUAL     50
RELEVOS        50
ALEVIN         44
Carrera        42
CAMINADA       41
CADETE         41
Cross          40
CORTA          39
POPULAR        39
Name: count, dtype: int64


### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, xipgroc, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`. `fuente` es una constante ("sportmaniacs") para identificar el origen al concatenar las tablas. `municipio`/`provincia` ya vienen directos de la fuente (`ciutat`/`provincia`); `comarca` no, así que la geocodificamos a partir de `municipio`+`provincia` (igual que en buscametas). Lo que es propio solo de sportmaniacs (`finisher_desconocido`, `id`, `modalidad`) va al final.

In [8]:
# Geocodificamos "comarca" a partir de "municipio"+"provincia" (igual que
# en buscametas): aquí ya tenemos municipio y provincia limpios de la
# fuente, así que solo falta la comarca. Los eventos están repartidos por
# toda España (no una sola comunidad autónoma), así que incluimos la
# provincia en la consulta para desambiguar. Como en buscametas/
# cronofinisher, "county" (comarca) de OpenStreetMap sale vacío a menudo
# fuera de Catalunya/Galicia — no es un fallo del código.
import csv
import time


def geocodificar_comarcas(curses_limpio, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_com = out_path / "sportmaniacs_comarcas.csv"

    pares = (
        curses_limpio[["municipio", "provincia"]]
        .drop_duplicates()
        .dropna(subset=["municipio"])
    )

    cache = {}
    if csv_com.exists():
        prev = pd.read_csv(csv_com, dtype=str)
        cache = {(r["municipio"], r["provincia"]): r["comarca"] for _, r in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} municipios ya geocodificados")

    geolocator = Nominatim(user_agent="sportmaniacs_comarcas_claudia")

    pendientes = [
        (m, p) for m, p in pares.itertuples(index=False)
        if (m, p) not in cache
    ]
    print(f"Municipios a geocodificar: {len(pendientes)} (de {len(pares)} únicos)")

    write_header = not csv_com.exists()
    with open(csv_com, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["municipio", "provincia", "comarca"])
        if write_header:
            writer.writeheader()

        for i, (municipio, provincia) in enumerate(pendientes, 1):
            comarca = None
            try:
                query = f"{municipio}, {provincia}, España" if pd.notna(provincia) else f"{municipio}, España"
                loc = geolocator.geocode(
                    query, exactly_one=True, country_codes="es", addressdetails=True, timeout=10,
                )
                if loc:
                    comarca = loc.raw.get("address", {}).get("county")
            except GeopyError as e:
                print(f"  [{municipio}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{municipio}] ERROR inesperado: {e}")

            writer.writerow({"municipio": municipio, "provincia": provincia, "comarca": comarca})
            f.flush()
            cache[(municipio, provincia)] = comarca

            if i % 50 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)

    print(f"CSV de comarcas: {csv_com.resolve()}")
    return cache


OUT_DIR = Path("../../data/raw/sportmaniacs")
_comarcas = geocodificar_comarcas(curses_limpio, out_dir=OUT_DIR)
curses_limpio["comarca"] = curses_limpio.apply(
    lambda row: _comarcas.get((row["municipio"], row["provincia"])), axis=1
)

print("Filas con comarca:", curses_limpio["comarca"].notna().sum(), "de", len(curses_limpio))
curses_limpio[["municipio", "provincia", "comarca"]].drop_duplicates().sample(15, random_state=0)

Checkpoint: 1651 municipios ya geocodificados
Municipios a geocodificar: 0 (de 1651 únicos)
CSV de comarcas: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\sportmaniacs_data\sportmaniacs_comarcas.csv
Filas con comarca: 8060 de 20844


,municipio,provincia,comarca
806,Borriol,Castellón,la Plana Alta
2374,Brozas,Cáceres,NaN
18544,Güeñes - Alegría-Dulantzi,Bizkaia,NaN
19689,Fuente de Pedro Naharro,Cuenca,NaN
20158,Adeje,Santa Cruz de Tenerife,NaN
4852,Almonacid de Toledo,Toledo,NaN
2443,Luarca,Asturias,NaN
7630,Fontellas,Navarra,Ribera / Erribera
4128,Caudete,Albacete,NaN
946,Sena,Huesca,Los Monegros


In [9]:
# Añadimos "fuente" (constante, para identificar el origen al concatenar
# con las otras 8 tablas) y "dia_semana" (derivado de "fecha"), y
# reordenamos las columnas para que el esquema común (fuente,
# nombre_carrera, fecha, dia_semana, distancia, tipo_modalidad, publico,
# finisher_d, finisher_h, municipio, comarca, provincia) quede igual en
# las 9 fuentes, dejando lo propio de sportmaniacs (finisher_desconocido,
# id, modalidad) al final.
curses_limpio["fuente"] = "sportmaniacs"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
     "finisher_desconocido", "id", "modalidad"]
]
curses_limpio.columns.tolist()

['fuente',
 'nombre_carrera',
 'fecha',
 'dia_semana',
 'distancia',
 'tipo_modalidad',
 'publico',
 'finisher_d',
 'finisher_h',
 'municipio',
 'comarca',
 'provincia',
 'finisher_desconocido',
 'id',
 'modalidad']

In [10]:
# Vista final de la tabla ya limpia y clasificada
print("Columnas:", list(curses_limpio.columns))
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
curses_limpio.sample(15)

Columnas: ['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia', 'finisher_desconocido', 'id', 'modalidad']
Filas x columnas: (20844, 15)

fuente                          object
nombre_carrera                  object
fecha                   datetime64[ns]
dia_semana                      object
distancia                      float64
tipo_modalidad                  object
publico                         object
finisher_d                       int64
finisher_h                       int64
municipio                       object
comarca                         object
provincia                       object
finisher_desconocido             int64
id                              object
modalidad                       object
dtype: object

Cruce tipo_modalidad x publico:
publico          Absoluta/General  Cadete/Juvenil  Elite  Equipos  Infantil  \
tipo_modalidad                                

,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia,finisher_desconocido,id,modalidad
1314,sportmaniacs,DUATLÓ CROSS SON NEGRE,2021-05-15,Sábado,0.0,Multidisciplina,Infantil,1,9,Felanitx,Migjorn,Baleares (Illes),0,609f7c04-3b24-43f0-9f7a-0b3aac1f12cd,Pre-benjamin
1265,sportmaniacs,II Volta a Peu per la discapacitat Ciutat de V...,2020-10-31,Sábado,0.0,Otros,Otros,81,185,Valencia,Comarca de València,Valencia,0,5fb238f3-bc48-417e-b39c-071cac1f125e,7000m
14333,sportmaniacs,I HYBRID RACE AGAETE 2026,2026-01-31,Sábado,0.0,Otros,Absoluta/General,19,0,Agaete,NaN,Las Palmas,0,6977b3be-bdac-4d5c-9298-467aac1f2621,PAREJAS FEMENINA
19975,sportmaniacs,Santander Triathlon Series - Tarragona 2018,2018-08-05,Domingo,0.0,Multidisciplina,Absoluta/General,0,3,Tarragona,Tarragonès,Tarragona,8,5b3e2828-ac74-4a7f-92e6-4512ac1f160f,Sprint Relevos
16519,sportmaniacs,XX TRIATLÓN OLÍMPICO DE SEVILLA,2016-05-29,Domingo,0.0,Multidisciplina,Absoluta/General,56,427,Sevilla,NaN,Sevilla,0,5697c569-1314-421a-8cd5-4982bc5ffd28,XX TRIATLON OLIMPICO DE SEVILLA
10734,sportmaniacs,XIII Media Maratón y X Cross San Isidro 2024,2024-05-05,Domingo,21.0,road running,Absoluta/General,12,106,Puebla de la Calzada,NaN,Badajoz,0,66375792-ce9c-44e6-ae46-eed1ac1f2a98,Media Maratón (21km)
4827,sportmaniacs,Carrera Parque de Miraflores,2022-05-29,Domingo,0.0,road running,Cadete/Juvenil,13,21,Sevilla,NaN,Sevilla,0,629368ec-3268-42ae-b037-4773ac1f0922,CADETES CARRERA PARQUE DE MIRAFLORES
14561,sportmaniacs,Backyard Cayón 2026,2026-03-21,Sábado,0.0,Otros,Absoluta/General,18,64,Santa Maráa de Cayón,NaN,Cantabria,0,69c056be-9680-4a68-a7cf-db16ac1f1294,Vuelta 16
10249,sportmaniacs,VI CARRERA DESAFÍO MTB-SPG SAN PEDRO DE GAÍLLOS,2024-03-02,Sábado,75.0,Ciclismo y btt,Absoluta/General,1,29,San Pedro de Gaállos,NaN,Segovia,0,65e5ce59-8548-4eab-8f9d-41a6ac1f03cd,DESAFIO 75 KM-EBIKE
6133,sportmaniacs,FARINATO RACE MADRID 2023,2023-04-15,Sábado,0.0,Otros,Absoluta/General,12,7,Madrid,NaN,Madrid,0,643d0883-891c-42b0-84f5-066dac1f2277,JOVENES FARINATOS


In [11]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/sportmaniacs/DF_SPORTMANIACS_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\sportmaniacs_data\DF_SPORTMANIACS_LIMPIO.csv
